In [1]:
import polars as pl
import gc

In [2]:
print("--- 1. Initializing Lazy Loader ---")
# Load the outlier-handled data without crashing RAM
file_path = '../data/processed/train_data_outliers_handled.parquet'
lf = pl.scan_parquet(file_path)

--- 1. Initializing Lazy Loader ---


In [3]:
print("--- 2. Defining Safe Math Variables ---")
# Helper function: If value is -999.0, treat as null. Otherwise keep value.
def safe(col_name):
    return pl.when(pl.col(col_name) == -999.0).then(None).otherwise(pl.col(col_name))

# 0.0001 epsilon prevents division-by-zero crashes
eps = 0.0001

--- 2. Defining Safe Math Variables ---


In [4]:
print("--- 3. Engineering Financial Signals ---")
# We calculate the features, then immediately fill any resulting nulls back to -999.0
fe_exprs = [
    # 1. UTILIZATION & RATIOS: Payment to Balance ratio
    # High payment relative to balance is safe; low payment to high balance is risky.
    (safe('P_2') / (safe('B_1') + eps)).fill_null(-999.0).alias('FE_Pay_to_Balance_Ratio'),
    
    # 2. DEBT CAPACITY: Balance (B_1) relative to Credit Limit/Safety (B_2)
    # Higher ratio means they are maxing out their available capacity.
    (safe('B_1') / (safe('B_2') + eps)).fill_null(-999.0).alias('FE_Debt_Utilization'),
    
    # 3. DELINQUENCY SEVERITY: Sum of major late-payment indicators
    # Combining the highly correlated D features into a single massive warning flag.
    (safe('D_39') + safe('D_44') + safe('D_48')).fill_null(-999.0).alias('FE_Delinquency_Severity'),
    
    # 4. INTERACTIONS: Multiplying strongly opposing signals
    # P_2 (safe) * B_9 (risky). If both are high, it creates a unique risk profile.
    (safe('P_2') * safe('B_9')).fill_null(-999.0).alias('FE_Payment_Risk_Interaction'),
    
    # 5. SPEND INTENSITY: Recent spend (S_3) vs Historical spend pattern
    (safe('S_3') / (safe('S_22') + eps)).fill_null(-999.0).alias('FE_Spend_Intensity')
]

# Apply the new features to our LazyFrame
lf_fe = lf.with_columns(fe_exprs)

--- 3. Engineering Financial Signals ---


In [5]:
print("--- 4. Executing and Saving ---")
# Define the new save path
save_path = '../data/processed/train_data_fe_completed.parquet'

# sink_parquet streams the data chunk-by-chunk from the hard drive, 
# applies the math, and writes it directly to the new file. Zero RAM crashes!
lf_fe.sink_parquet(save_path)

print(f" Feature Engineering Complete!")
print(f"Dataset saved with new financial signals at: {save_path}")

--- 4. Executing and Saving ---
 Feature Engineering Complete!
Dataset saved with new financial signals at: ../data/processed/train_data_fe_completed.parquet
